In [2]:
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
print(cartopy.__version__)

ModuleNotFoundError: No module named 'cartopy'

In [1]:
# ============================================================
# Χάρτης θέσης παλιρροιογράφου Ge.In. NOA / HL-NTWC
# - Prompt για lat, lon, κωδικό ΧΧΧΧ, και paths των 2 logos
# - Κύριος χάρτης γύρω από τη θέση
# - Inset χάρτης Ελλάδας κάτω αριστερά
# - Πλαίσιο logos + υπόμνημα πάνω δεξιά
# - Τίτλος: Παλιρροιογράφος XXXX
# ============================================================

import os
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
except ImportError as e:
    raise ImportError(
        "Δεν βρέθηκε το cartopy. Σε Anaconda εγκατάσταση π.χ.:\n"
        "conda install -c conda-forge cartopy"
    ) from e


# -----------------------------
# Ρυθμίσεις χάρτη
# -----------------------------
MAP_WIDTH_KM = 12.0
MAP_HEIGHT_KM = 12.0
FIG_W = 13
FIG_H = 10
OUTPUT_DPI = 300

# Ελλάδα - extent inset
GREECE_EXTENT = [19.0, 30.5, 34.0, 42.5]  # [lon_min, lon_max, lat_min, lat_max]


# -----------------------------
# Βοηθητικές συναρτήσεις
# -----------------------------
def clean_path(p):
    return p.strip().strip('"').strip("'")

def validate_lat_lon(lat, lon):
    if not (-90.0 <= lat <= 90.0):
        raise ValueError("Το latitude πρέπει να είναι στο [-90, 90].")
    if not (-180.0 <= lon <= 180.0):
        raise ValueError("Το longitude πρέπει να είναι στο [-180, 180].")

def km_to_deg(lat_deg, width_km, height_km):
    """
    Μετατροπή km -> degrees γύρω από δοσμένο γεωγραφικό πλάτος.
    Επαρκές για τοπικό χάρτη λίγων km.
    """
    lat_rad = math.radians(lat_deg)

    # προσεγγιστικά μήκη 1 degree
    km_per_deg_lat = 110.574
    km_per_deg_lon = 111.320 * max(math.cos(lat_rad), 1e-8)

    dlat = (height_km / 2.0) / km_per_deg_lat
    dlon = (width_km / 2.0) / km_per_deg_lon
    return dlat, dlon

def load_logo(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Δεν βρέθηκε το αρχείο logo: {path}")
    img = Image.open(path).convert("RGBA")
    return np.asarray(img)

def compute_zoom(img_array, target_w_px=260, target_h_px=110):
    h, w = img_array.shape[:2]
    zoom_w = target_w_px / w
    zoom_h = target_h_px / h
    return min(zoom_w, zoom_h)

def add_logo(ax_panel, img_array, xy_axes, target_w_px=260, target_h_px=110):
    zoom = compute_zoom(img_array, target_w_px=target_w_px, target_h_px=target_h_px)
    oi = OffsetImage(img_array, zoom=zoom)
    ab = AnnotationBbox(
        oi,
        xy_axes,
        xycoords=ax_panel.transAxes,
        frameon=False,
        box_alignment=(0.5, 0.5),
        zorder=5
    )
    ax_panel.add_artist(ab)

def create_map(lat, lon, tg_code, noa_logo_path, hlntwc_logo_path, out_path):
    validate_lat_lon(lat, lon)

    noa_logo = load_logo(noa_logo_path)
    hl_logo = load_logo(hlntwc_logo_path)

    dlat, dlon = km_to_deg(lat, MAP_WIDTH_KM, MAP_HEIGHT_KM)

    proj = ccrs.PlateCarree()

    fig = plt.figure(figsize=(FIG_W, FIG_H))

    # =========================================================
    # Κύριος χάρτης
    # =========================================================
    ax = fig.add_axes([0.05, 0.07, 0.90, 0.88], projection=proj)
    ax.set_extent([lon - dlon, lon + dlon, lat - dlat, lat + dlat], crs=proj)

    ax.add_feature(cfeature.LAND, facecolor="#f2efe9", zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor="#dbeaf5", zorder=0)
    ax.add_feature(cfeature.COASTLINE.with_scale("10m"), linewidth=1.0, zorder=2)
    ax.add_feature(cfeature.BORDERS.with_scale("10m"), linewidth=0.5, zorder=2)

    gl = ax.gridlines(
        crs=proj,
        draw_labels=True,
        linewidth=0.5,
        color="gray",
        alpha=0.6,
        linestyle="--"
    )
    gl.top_labels = False
    gl.right_labels = False

    ax.plot(
        lon, lat,
        marker="*",
        markersize=16,
        markerfacecolor="red",
        markeredgecolor="black",
        transform=proj,
        zorder=10
    )

    ax.text(
        lon, lat,
        f"  {tg_code}",
        transform=proj,
        fontsize=11,
        fontweight="bold",
        va="center",
        ha="left",
        zorder=10,
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=1.5)
    )

    ax.set_title(f"Παλιρροιογράφος {tg_code}", fontsize=18, fontweight="bold", pad=12)

    # =========================================================
    # Inset Ελλάδας - κάτω αριστερά
    # =========================================================
    ax_inset = fig.add_axes([0.08, 0.10, 0.22, 0.22], projection=proj)
    ax_inset.set_extent(GREECE_EXTENT, crs=proj)

    ax_inset.add_feature(cfeature.LAND, facecolor="#f2efe9", zorder=0)
    ax_inset.add_feature(cfeature.OCEAN, facecolor="#dbeaf5", zorder=0)
    ax_inset.add_feature(cfeature.COASTLINE.with_scale("10m"), linewidth=0.7, zorder=2)
    ax_inset.add_feature(cfeature.BORDERS.with_scale("10m"), linewidth=0.4, zorder=2)

    ax_inset.plot(
        lon, lat,
        marker="*",
        markersize=10,
        markerfacecolor="red",
        markeredgecolor="black",
        transform=proj,
        zorder=10
    )

    # rectangle της κύριας κάλυψης
    rect = Rectangle(
        (lon - dlon, lat - dlat),
        2 * dlon,
        2 * dlat,
        fill=False,
        edgecolor="red",
        linewidth=1.2,
        transform=proj,
        zorder=9
    )
    ax_inset.add_patch(rect)

    ax_inset.set_xticks([])
    ax_inset.set_yticks([])
    ax_inset.set_title("Θέση στην Ελλάδα", fontsize=10, pad=4)

    # =========================================================
    # Πάνελ logos + υπόμνημα - πάνω δεξιά
    # =========================================================
    ax_panel = fig.add_axes([0.69, 0.63, 0.26, 0.25])
    ax_panel.set_facecolor("white")
    ax_panel.set_xlim(0, 1)
    ax_panel.set_ylim(0, 1)
    ax_panel.set_xticks([])
    ax_panel.set_yticks([])
    for spine in ax_panel.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.0)
        spine.set_edgecolor("black")

    # logos στην κορυφή
    add_logo(ax_panel, noa_logo, (0.28, 0.73), target_w_px=150, target_h_px=80)
    add_logo(ax_panel, hl_logo,  (0.74, 0.73), target_w_px=150, target_h_px=80)

    # υπόμνημα κάτω από τα logos
    ax_panel.plot(
        0.12, 0.22,
        marker="*",
        markersize=14,
        markerfacecolor="red",
        markeredgecolor="black"
    )
    ax_panel.text(
        0.20, 0.22,
        "Θέση παλιρροιογράφου",
        fontsize=10,
        va="center",
        ha="left"
    )

    # =========================================================
    # Αποθήκευση / εμφάνιση
    # =========================================================
    plt.savefig(out_path, dpi=OUTPUT_DPI, bbox_inches="tight")
    plt.show()

    print(f"Ο χάρτης αποθηκεύτηκε στο: {os.path.abspath(out_path)}")


# -----------------------------
# Prompt προς χρήστη
# -----------------------------
lat = float(input("Δώσε latitude (decimal degrees, GRS80): ").strip())
lon = float(input("Δώσε longitude (decimal degrees, GRS80): ").strip())
tg_code = input("Δώσε τον κωδικό παλιρροιογράφου XXXX (π.χ. ASTY): ").strip().upper()

noa_logo_path = clean_path(input("Δώσε path για το logo του National Observatory of Athens: "))
hlntwc_logo_path = clean_path(input("Δώσε path για το logo του Hellenic National Tsunami Warning Centre (HL-NTWC): "))

out_path = clean_path(input(f"Δώσε path εξόδου [ENTER για map_{tg_code}.png]: ").strip())
if out_path == "":
    out_path = f"map_{tg_code}.png"

create_map(
    lat=lat,
    lon=lon,
    tg_code=tg_code,
    noa_logo_path=noa_logo_path,
    hlntwc_logo_path=hlntwc_logo_path,
    out_path=out_path
)

ImportError: Δεν βρέθηκε το cartopy. Σε Anaconda εγκατάσταση π.χ.:
conda install -c conda-forge cartopy